In [0]:
-- Set the default catalog and schema for this notebook.
-- All unqualified table references resolve to market_risk_dev.gold
USE CATALOG market_risk_dev;
USE SCHEMA gold;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- gold.var_daily
-- Purpose : Daily VaR, 10-day VaR and Expected Shortfall by desk and asset class
-- Method  : Historical simulation using 260 trading days of price returns
-- Grain   : One row per desk per asset_class per calculation_date
-- Source  : silver.positions_enriched + silver.prices_cleaned
-- Regulatory context:
--   var_95_usd         — internal management reporting
--   var_97_5_usd       — Basel III / FRTB standard confidence level
--   var_99_usd         — senior management / board reporting
--   var_10day_usd      — Basel III regulatory capital (1-day VaR × √10)
--   es_97_5_usd        — FRTB Expected Shortfall, primary regulatory measure
-- Notes   : ZORDER candidate: (calculation_date, desk) — add in Phase 7
-- ─────────────────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE market_risk_dev.gold.var_daily (
  calculation_date      DATE    NOT NULL COMMENT 'Date VaR was calculated',
  desk                  STRING  NOT NULL COMMENT 'Trading desk name',
  asset_class           STRING  NOT NULL COMMENT 'Asset class: FX, Equity, Rates, Credit',
  position_count        INT     NOT NULL COMMENT 'Number of positions in this desk/asset_class group',
  total_notional_usd    DOUBLE  NOT NULL COMMENT 'Total notional exposure in USD for this group',
  var_95_usd            DOUBLE  NOT NULL COMMENT '1-day VaR at 95th percentile — 1 in 20 days loss threshold. Internal management use.',
  var_97_5_usd          DOUBLE  NOT NULL COMMENT '1-day VaR at 97.5th percentile — Basel III / FRTB standard confidence level.',
  var_99_usd            DOUBLE  NOT NULL COMMENT '1-day VaR at 99th percentile — 1 in 100 days loss threshold. Senior management use.',
  var_10day_95_usd      DOUBLE  NOT NULL COMMENT '10-day VaR at 95th percentile. Derived as var_95 × √10. Basel III regulatory capital input.',
  var_10day_97_5_usd    DOUBLE  NOT NULL COMMENT '10-day VaR at 97.5th percentile. Derived as var_97_5 × √10. Primary Basel III capital measure.',
  var_10day_99_usd      DOUBLE  NOT NULL COMMENT '10-day VaR at 99th percentile. Derived as var_99 × √10.',
  es_97_5_usd           DOUBLE  NOT NULL COMMENT 'Expected Shortfall at 97.5th percentile. Average loss on days exceeding var_97_5. FRTB primary regulatory measure.',
  avg_daily_return_usd  DOUBLE           COMMENT 'Average daily P&L across all 260 scenarios for this group',
  worst_day_loss_usd    DOUBLE           COMMENT 'Largest single-day loss in the 260-day history',
  best_day_gain_usd     DOUBLE           COMMENT 'Largest single-day gain in the 260-day history',
  scenario_count        INT     NOT NULL COMMENT 'Number of historical scenarios used. Should be ~260.',
  _gold_loaded_at       TIMESTAMP        COMMENT 'Timestamp when row was loaded into Gold'
)
USING DELTA
COMMENT 'Daily VaR, 10-day VaR and Expected Shortfall by desk and asset class. Historical simulation method. One row per desk per asset_class per calculation_date.'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite'   = 'true',
  'delta.autoOptimize.autoCompact'     = 'true',
  'delta.logRetentionDuration'         = 'interval 90 days',
  'delta.deletedFileRetentionDuration' = 'interval 90 days'
);

In [0]:
-- MERGE ensures history accumulates correctly.
-- On rerun of the same date: WHEN MATCHED updates existing rows.
-- On new date: WHEN NOT MATCHED inserts new rows.
-- Historical rows from previous dates are never touched.
MERGE INTO market_risk_dev.gold.var_daily AS target
USING (

WITH

-- Step 1: Calculate daily returns for each ticker.
--         Return = (today's close - yesterday's close) / yesterday's close
--         LAG() retrieves the previous row's close price within each
--         ticker partition ordered by date ascending.
daily_returns AS (
  SELECT
    ticker,
    price_date,
    close_price,
    LAG(close_price) OVER (
      PARTITION BY ticker
      ORDER BY price_date ASC
    ) AS prev_close_price
  FROM market_risk_dev.silver.prices_cleaned
),

-- Step 2: Calculate percentage return and filter to valid rows only.
--         Null return = first row per ticker (no previous day available).
--         Zero prev_close excluded to prevent division by zero.
--         Date filter keeps only the most recent 260 trading days —
--         the standard lookback window for historical simulation VaR.
return_pct AS (
  SELECT
    ticker,
    price_date,
    (close_price - prev_close_price) / prev_close_price AS daily_return_pct
  FROM daily_returns
  WHERE
    prev_close_price IS NOT NULL
    AND prev_close_price > 0
    AND price_date >= (
      SELECT DATE_SUB(MAX(price_date), 365)
      FROM market_risk_dev.silver.prices_cleaned
    )
),

-- Step 3: Join positions to return scenarios.
--         Each position generates one scenario P&L per historical trading day.
--         P&L = notional_usd × daily_return_pct × direction_multiplier
--         LONG positions lose when prices fall (multiplier=1, return negative).
--         SHORT positions gain when prices fall (multiplier=-1, return negative
--         × -1 = positive P&L).
position_scenarios AS (
  SELECT
    p.desk,
    p.asset_class,
    p.trade_id,
    p.notional_usd,
    p.direction_multiplier,
    r.price_date                                         AS scenario_date,
    ROUND(
      p.notional_usd * r.daily_return_pct * p.direction_multiplier,
      2
    )                                                    AS scenario_pnl_usd
  FROM market_risk_dev.silver.positions_enriched p
  INNER JOIN return_pct                          r
    ON p.ticker = r.ticker
),

-- Step 4: Aggregate scenario P&L to desk + asset_class level.
--         Each row represents one portfolio-level scenario
--         for one desk + asset_class on one historical date.
portfolio_scenarios AS (
  SELECT
    desk,
    asset_class,
    scenario_date,
    SUM(scenario_pnl_usd)    AS portfolio_pnl_usd,
    COUNT(DISTINCT trade_id) AS position_count,
    SUM(notional_usd)        AS total_notional_usd
  FROM position_scenarios
  GROUP BY
    desk,
    asset_class,
    scenario_date
),

-- Step 5: Calculate VaR at three confidence levels using PERCENTILE_CONT.
--         We use left tail percentiles:
--           0.05  → 95th percentile VaR (5% of days are worse)
--           0.025 → 97.5th percentile VaR (2.5% of days are worse)
--           0.01  → 99th percentile VaR (1% of days are worse)
--         ABS() converts negative loss values to positive VaR figures.
--         VaR is always reported as a positive number by convention.
var_base AS (
  SELECT
    desk,
    asset_class,
    MAX(position_count)                                  AS position_count,
    MAX(total_notional_usd)                              AS total_notional_usd,
    COUNT(*)                                             AS scenario_count,
    ABS(
      PERCENTILE_CONT(0.05) WITHIN GROUP (
        ORDER BY portfolio_pnl_usd ASC
      )
    )                                                    AS var_95_usd,
    ABS(
      PERCENTILE_CONT(0.025) WITHIN GROUP (
        ORDER BY portfolio_pnl_usd ASC
      )
    )                                                    AS var_97_5_usd,
    ABS(
      PERCENTILE_CONT(0.01) WITHIN GROUP (
        ORDER BY portfolio_pnl_usd ASC
      )
    )                                                    AS var_99_usd,
    AVG(portfolio_pnl_usd)                               AS avg_daily_return_usd,
    MIN(portfolio_pnl_usd)                               AS worst_day_loss_usd,
    MAX(portfolio_pnl_usd)                               AS best_day_gain_usd
  FROM portfolio_scenarios
  GROUP BY
    desk,
    asset_class
),

-- Step 6: Calculate Expected Shortfall at 97.5%.
--         ES = average of all losses WORSE than the VaR threshold.
--         Join back to portfolio_scenarios to get individual scenario P&Ls
--         then filter to only scenarios in the tail beyond VaR 97.5.
--         ES is calculated separately because it needs row-level scenario
--         data, not just the percentile value.
es_calculation AS (
  SELECT
    ps.desk,
    ps.asset_class,
    ABS(AVG(ps.portfolio_pnl_usd)) AS es_97_5_usd
  FROM portfolio_scenarios ps
  INNER JOIN var_base       vb
    ON  ps.desk        = vb.desk
    AND ps.asset_class = vb.asset_class
  -- Keep only scenarios worse than the 97.5% VaR threshold
  WHERE ps.portfolio_pnl_usd < -vb.var_97_5_usd
  GROUP BY
    ps.desk,
    ps.asset_class
),

-- Step 7: Combine VaR and ES.
--         10-day VaR derived using the square root of time rule:
--         10-day VaR = 1-day VaR × √10 = 1-day VaR × 3.16228
--         Assumption: daily returns are independent and identically distributed.
--         Known limitation: this assumption breaks down during market stress.
combined AS (
  SELECT
    vb.desk,
    vb.asset_class,
    vb.position_count,
    vb.total_notional_usd,
    vb.var_95_usd,
    vb.var_97_5_usd,
    vb.var_99_usd,
    ROUND(vb.var_95_usd   * SQRT(10), 2)                AS var_10day_95_usd,
    ROUND(vb.var_97_5_usd * SQRT(10), 2)                AS var_10day_97_5_usd,
    ROUND(vb.var_99_usd   * SQRT(10), 2)                AS var_10day_99_usd,
    -- Fall back to var_97_5 if no tail scenarios exist for ES calculation
    COALESCE(ROUND(es.es_97_5_usd, 2), vb.var_97_5_usd) AS es_97_5_usd,
    vb.avg_daily_return_usd,
    vb.worst_day_loss_usd,
    vb.best_day_gain_usd,
    vb.scenario_count
  FROM var_base            vb
  LEFT JOIN es_calculation es
    ON  vb.desk        = es.desk
    AND vb.asset_class = es.asset_class
)

SELECT
  CURRENT_DATE()                       AS calculation_date,
  desk,
  asset_class,
  CAST(position_count  AS INT)         AS position_count,
  ROUND(total_notional_usd, 2)         AS total_notional_usd,
  ROUND(var_95_usd, 2)                 AS var_95_usd,
  ROUND(var_97_5_usd, 2)               AS var_97_5_usd,
  ROUND(var_99_usd, 2)                 AS var_99_usd,
  var_10day_95_usd,
  var_10day_97_5_usd,
  var_10day_99_usd,
  es_97_5_usd,
  ROUND(avg_daily_return_usd, 2)       AS avg_daily_return_usd,
  ROUND(worst_day_loss_usd, 2)         AS worst_day_loss_usd,
  ROUND(best_day_gain_usd, 2)          AS best_day_gain_usd,
  CAST(scenario_count AS INT)          AS scenario_count,
  current_timestamp()                  AS _gold_loaded_at
FROM combined ) AS source
ON  target.desk = source.desk
AND target.asset_class = source.asset_class
AND target.calculation_date = source.calculation_date
WHEN MATCHED THEN UPDATE SET
  target.position_count       = source.position_count,
  target.total_notional_usd   = source.total_notional_usd,
  target.var_95_usd           = source.var_95_usd,
  target.var_97_5_usd         = source.var_97_5_usd,
  target.var_99_usd           = source.var_99_usd,
  target.var_10day_95_usd     = source.var_10day_95_usd,
  target.var_10day_97_5_usd   = source.var_10day_97_5_usd,
  target.var_10day_99_usd     = source.var_10day_99_usd,
  target.es_97_5_usd          = source.es_97_5_usd,
  target.avg_daily_return_usd = source.avg_daily_return_usd,
  target.worst_day_loss_usd   = source.worst_day_loss_usd,
  target.best_day_gain_usd    = source.best_day_gain_usd,
  target.scenario_count       = source.scenario_count,
  target._gold_loaded_at      = source._gold_loaded_at
WHEN NOT MATCHED THEN INSERT (
  calculation_date,
  desk,
  asset_class,
  position_count,
  total_notional_usd,
  var_95_usd,
  var_97_5_usd,
  var_99_usd,
  var_10day_95_usd,
  var_10day_97_5_usd,
  var_10day_99_usd,
  es_97_5_usd,
  avg_daily_return_usd,
  worst_day_loss_usd,
  best_day_gain_usd,
  scenario_count,
  _gold_loaded_at
)
VALUES (
  source.calculation_date,
  source.desk,
  source.asset_class,
  source.position_count,
  source.total_notional_usd,
  source.var_95_usd,
  source.var_97_5_usd,
  source.var_99_usd,
  source.var_10day_95_usd,
  source.var_10day_97_5_usd,
  source.var_10day_99_usd,
  source.es_97_5_usd,
  source.avg_daily_return_usd,
  source.worst_day_loss_usd,
  source.best_day_gain_usd,
  source.scenario_count,
  source._gold_loaded_at
);

In [0]:
-- Data quality check for gold.var_daily
WITH quality_summary AS (
  SELECT
    COUNT(*)                                                      AS total_rows,
    COUNT(DISTINCT desk)                                          AS unique_desks,
    COUNT(DISTINCT asset_class)                                   AS unique_asset_classes,
    COUNT(DISTINCT calculation_date)                              AS unique_dates,
    COUNT(CASE WHEN var_95_usd         IS NULL THEN 1 END)        AS null_var_95,
    COUNT(CASE WHEN var_97_5_usd       IS NULL THEN 1 END)        AS null_var_97_5,
    COUNT(CASE WHEN var_99_usd         IS NULL THEN 1 END)        AS null_var_99,
    COUNT(CASE WHEN var_10day_97_5_usd IS NULL THEN 1 END)        AS null_var_10day,
    COUNT(CASE WHEN es_97_5_usd        IS NULL THEN 1 END)        AS null_es,
    COUNT(CASE WHEN total_notional_usd IS NULL THEN 1 END)        AS null_notional,
    -- Monotonicity: VaR must increase with confidence level
    COUNT(CASE WHEN var_97_5_usd    < var_95_usd     THEN 1 END)  AS var_97_5_less_than_95,
    COUNT(CASE WHEN var_99_usd      < var_97_5_usd   THEN 1 END)  AS var_99_less_than_97_5,
    -- ES must always be >= VaR at same confidence level
    COUNT(CASE WHEN es_97_5_usd     < var_97_5_usd   THEN 1 END)  AS es_less_than_var,
    -- 10-day VaR must be larger than 1-day VaR
    COUNT(CASE WHEN var_10day_97_5_usd < var_97_5_usd THEN 1 END) AS tenday_less_than_oneday,
    COUNT(CASE WHEN var_95_usd     <= 0 THEN 1 END)               AS non_positive_var_95,
    COUNT(CASE WHEN es_97_5_usd    <= 0 THEN 1 END)               AS non_positive_es,
    COUNT(CASE WHEN scenario_count  < 200 THEN 1 END)             AS low_scenario_count
  FROM market_risk_dev.gold.var_daily
)

SELECT
  total_rows,
  unique_desks,
  unique_asset_classes,
  unique_dates,
  null_var_95,
  null_var_97_5,
  null_var_99,
  null_var_10day,
  null_es,
  null_notional,
  var_97_5_less_than_95,
  var_99_less_than_97_5,
  es_less_than_var,
  tenday_less_than_oneday,
  non_positive_var_95,
  non_positive_es,
  low_scenario_count,
  CASE
    WHEN null_var_95             > 0 THEN 'FAIL — null var_95 found'
    WHEN null_var_97_5           > 0 THEN 'FAIL — null var_97_5 found'
    WHEN null_var_99             > 0 THEN 'FAIL — null var_99 found'
    WHEN null_var_10day          > 0 THEN 'FAIL — null var_10day found'
    WHEN null_es                 > 0 THEN 'FAIL — null es_97_5 found'
    WHEN null_notional           > 0 THEN 'FAIL — null notional found'
    WHEN var_97_5_less_than_95   > 0 THEN 'FAIL — var_97_5 less than var_95'
    WHEN var_99_less_than_97_5   > 0 THEN 'FAIL — var_99 less than var_97_5'
    WHEN es_less_than_var        > 0 THEN 'FAIL — ES less than VaR at same confidence level'
    WHEN tenday_less_than_oneday > 0 THEN 'FAIL — 10-day VaR less than 1-day VaR'
    WHEN non_positive_var_95     > 0 THEN 'FAIL — non-positive var_95'
    WHEN non_positive_es         > 0 THEN 'FAIL — non-positive ES'
    WHEN low_scenario_count      > 0 THEN 'WARN — fewer than 200 scenarios used'
    ELSE 'PASS'
  END AS quality_status
FROM quality_summary;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- gold.pnl_attribution
-- Purpose : Daily PnL attribution — actual vs hypothetical by desk
-- Grain   : One row per desk per pnl_date
-- Source  : silver.positions_enriched + silver.prices_cleaned
-- Notes   : Hypothetical PnL uses only market price moves (clean P&L).
--           Unexplained PnL = actual - hypothetical.
--           Large unexplained values indicate unmodelled risk or booking errors.
-- ─────────────────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE market_risk_dev.gold.pnl_attribution (
  pnl_date              DATE    NOT NULL COMMENT 'Date PnL was calculated',
  desk                  STRING  NOT NULL COMMENT 'Trading desk name',
  hypothetical_pnl_usd  DOUBLE  NOT NULL COMMENT 'PnL explained purely by market price moves — clean P&L',
  actual_pnl_usd        DOUBLE  NOT NULL COMMENT 'Total PnL including all factors',
  unexplained_pnl_usd   DOUBLE  NOT NULL COMMENT 'Actual minus hypothetical. Large values are a red flag.',
  position_count        INT     NOT NULL COMMENT 'Number of positions contributing to PnL',
  long_pnl_usd          DOUBLE  NOT NULL COMMENT 'PnL contribution from LONG positions only',
  short_pnl_usd         DOUBLE  NOT NULL COMMENT 'PnL contribution from SHORT positions only',
  _gold_loaded_at       TIMESTAMP        COMMENT 'Timestamp when row was loaded into Gold'
)
USING DELTA
COMMENT 'Daily PnL attribution by desk. Actual vs hypothetical PnL with unexplained component. One row per desk per pnl_date.'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite'   = 'true',
  'delta.autoOptimize.autoCompact'     = 'true',
  'delta.logRetentionDuration'         = 'interval 90 days',
  'delta.deletedFileRetentionDuration' = 'interval 90 days'
);

In [0]:
MERGE INTO market_risk_dev.gold.pnl_attribution AS target
USING (
WITH

-- Step 1: Get the two most recent trading days per ticker.
--         DENSE_RANK() numbers dates descending per ticker:
--         rank 1 = most recent, rank 2 = previous trading day.
ranked_dates AS (
  SELECT
    ticker,
    price_date,
    close_price,
    DENSE_RANK() OVER (
      PARTITION BY ticker
      ORDER BY price_date DESC
    ) AS date_rank
  FROM market_risk_dev.silver.prices_cleaned
),

-- Step 2: Pivot the two ranked rows into two columns per ticker.
--         MAX(CASE...) avoids a self-join and produces one clean row
--         per ticker with both current and prior close prices.
current_and_prior_prices AS (
  SELECT
    ticker,
    MAX(CASE WHEN date_rank = 1 THEN close_price END) AS current_close,
    MAX(CASE WHEN date_rank = 2 THEN close_price END) AS prior_close,
    MAX(CASE WHEN date_rank = 1 THEN price_date  END) AS current_date
  FROM ranked_dates
  WHERE date_rank IN (1, 2)
  GROUP BY ticker
),

-- Step 3: Calculate the daily price return per ticker.
--         Null return = only one day of history exists for that ticker.
price_moves AS (
  SELECT
    ticker,
    current_date                                        AS price_date,
    current_close,
    prior_close,
    CASE
      WHEN prior_close IS NULL OR prior_close = 0 THEN NULL
      ELSE (current_close - prior_close) / prior_close
    END                                                 AS daily_return_pct
  FROM current_and_prior_prices
),

-- Step 4: Calculate P&L per position.
--         Hypothetical PnL = pure market move P&L.
--         Actual PnL = market move + small noise simulating unexplained
--         component from new trades, theta decay, and other real-world factors.
--         In production, actual PnL comes from the trade booking system.
position_pnl AS (
  SELECT
    p.desk,
    p.trade_id,
    p.direction,
    pm.price_date,
    ROUND(
      p.notional_usd * COALESCE(pm.daily_return_pct, 0) * p.direction_multiplier,
      2
    )                                                   AS hypothetical_pnl_usd,
    ROUND(
      p.notional_usd
      * COALESCE(pm.daily_return_pct, 0)
      * p.direction_multiplier
      * (1 + (HASH(p.trade_id) % 100) / 10000.0),
      2
    )                                                   AS actual_pnl_usd
  FROM market_risk_dev.silver.positions_enriched p
  LEFT JOIN price_moves                          pm
    ON p.ticker = pm.ticker
),

-- Step 5: Aggregate to desk level.
desk_pnl AS (
  SELECT
    desk,
    price_date                                          AS pnl_date,
    COUNT(DISTINCT trade_id)                            AS position_count,
    ROUND(SUM(hypothetical_pnl_usd), 2)                AS hypothetical_pnl_usd,
    ROUND(SUM(actual_pnl_usd), 2)                      AS actual_pnl_usd,
    ROUND(
      SUM(actual_pnl_usd) - SUM(hypothetical_pnl_usd),
      2
    )                                                   AS unexplained_pnl_usd,
    ROUND(
      SUM(CASE WHEN direction = 'LONG'
               THEN actual_pnl_usd ELSE 0 END),
      2
    )                                                   AS long_pnl_usd,
    ROUND(
      SUM(CASE WHEN direction = 'SHORT'
               THEN actual_pnl_usd ELSE 0 END),
      2
    )                                                   AS short_pnl_usd
  FROM position_pnl
  WHERE price_date IS NOT NULL
  GROUP BY
    desk,
    price_date
)

SELECT
  pnl_date,
  desk,
  hypothetical_pnl_usd,
  actual_pnl_usd,
  unexplained_pnl_usd,
  CAST(position_count AS INT) AS position_count,
  long_pnl_usd,
  short_pnl_usd,
  current_timestamp()         AS _gold_loaded_at
FROM desk_pnl
) AS source
ON  source.pnl_date = target.pnl_date 
AND source.desk = target.desk
WHEN MATCHED THEN 
  UPDATE SET 
    target.hypothetical_pnl_usd     = source.hypothetical_pnl_usd,
    target.actual_pnl_usd           = source.actual_pnl_usd,
    target.unexplained_pnl_usd      = source.unexplained_pnl_usd,
    target.position_count           = source.position_count,
    target.long_pnl_usd             = source.long_pnl_usd,
    target.short_pnl_usd            = source.short_pnl_usd,
    target._gold_loaded_at          = source._gold_loaded_at
WHEN NOT MATCHED THEN INSERT(
    pnl_date,
    desk,
    hypothetical_pnl_usd,
    actual_pnl_usd,
    unexplained_pnl_usd,
    position_count,
    long_pnl_usd,
    short_pnl_usd,
    _gold_loaded_at
  )
  VALUES (
    source.pnl_date,
    source.desk,
    source.hypothetical_pnl_usd,
    source.actual_pnl_usd,
    source.unexplained_pnl_usd,
    source.position_count,
    source.long_pnl_usd,
    source.short_pnl_usd,
    source._gold_loaded_at
  );

In [0]:
-- Data quality check for gold.pnl_attribution table
WITH quality_summary AS (
  SELECT
    COUNT(*)                                                      AS total_rows,
    COUNT(DISTINCT desk)                                          AS unique_desks,
    COUNT(DISTINCT pnl_date)                                      AS unique_dates,
    COUNT(CASE WHEN hypothetical_pnl_usd IS NULL THEN 1 END)      AS null_hypothetical,
    COUNT(CASE WHEN actual_pnl_usd       IS NULL THEN 1 END)      AS null_actual,
    COUNT(CASE WHEN unexplained_pnl_usd  IS NULL THEN 1 END)      AS null_unexplained,
    -- Arithmetic consistency: unexplained must equal actual minus hypothetical
    COUNT(
      CASE WHEN ABS(
        unexplained_pnl_usd - (actual_pnl_usd - hypothetical_pnl_usd)
      ) > 0.01 THEN 1 END
    )                                                             AS pnl_mismatch_rows
  FROM market_risk_dev.gold.pnl_attribution
)

SELECT
  total_rows,
  unique_desks,
  unique_dates,
  null_hypothetical,
  null_actual,
  null_unexplained,
  pnl_mismatch_rows,
  CASE
    WHEN null_hypothetical  > 0 THEN 'FAIL — null hypothetical_pnl found'
    WHEN null_actual        > 0 THEN 'FAIL — null actual_pnl found'
    WHEN null_unexplained   > 0 THEN 'FAIL — null unexplained_pnl found'
    WHEN pnl_mismatch_rows  > 0 THEN 'FAIL — unexplained PnL arithmetic mismatch'
    ELSE 'PASS'
  END AS quality_status
FROM quality_summary;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- gold.var_backtest
-- Purpose : VaR backtesting — counts exceptions where actual loss exceeded VaR
-- Grain   : One row per desk per backtest_date
-- Source  : gold.var_daily + gold.pnl_attribution
-- Regulatory context:
--   Basel traffic light (exceptions in rolling 250 trading days):
--   GREEN  — 0 to 4  — model accepted, no capital add-on
--   AMBER  — 5 to 9  — increased scrutiny, possible capital add-on
--   RED    — 10+     — model rejected, mandatory capital add-on
-- ─────────────────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE market_risk_dev.gold.var_backtest (
  backtest_date             DATE    NOT NULL COMMENT 'Date of the backtest observation',
  desk                      STRING  NOT NULL COMMENT 'Trading desk name',
  var_97_5_usd              DOUBLE  NOT NULL COMMENT '1-day VaR at 97.5th percentile used in this backtest',
  actual_loss_usd           DOUBLE  NOT NULL COMMENT 'Actual loss on this date. Positive = loss day. Negative = gain day.',
  is_exception              BOOLEAN NOT NULL COMMENT 'True when actual loss exceeded VaR — a backtesting exception',
  exception_magnitude       DOUBLE           COMMENT 'How much the actual loss exceeded VaR. Null when is_exception = false.',
  rolling_250d_exceptions   INT     NOT NULL COMMENT 'Count of exceptions in the rolling 250 trading day window — Basel standard',
  basel_zone                STRING  NOT NULL COMMENT 'Basel traffic light: GREEN (0-4 exceptions), AMBER (5-9), RED (10+)',
  model_status              STRING  NOT NULL COMMENT 'Model assessment: ACCEPTED, REVIEW, REJECTED',
  _gold_loaded_at           TIMESTAMP        COMMENT 'Timestamp when row was loaded into Gold'
)
USING DELTA
COMMENT 'VaR backtesting results by desk. Exception counts, rolling 250-day window, Basel traffic light zone. One row per desk per backtest_date.'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite'   = 'true',
  'delta.autoOptimize.autoCompact'     = 'true',
  'delta.logRetentionDuration'         = 'interval 90 days',
  'delta.deletedFileRetentionDuration' = 'interval 90 days'
);

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- KNOWN LIMITATION: rolling_250d_exceptions is calculated only across rows
-- in the current run, not across the full accumulated history in the target
-- table. This means the rolling count will not be accurate until this logic
-- is ported to dbt incremental in Phase 5, where the window function will
-- operate across the full history.
-- unique_key = [desk, backtest_date]
-- ───────────────────────────────────────────────────────────────────────────── 
MERGE INTO market_risk_dev.gold.var_backtest AS target
USING (
WITH

-- Step 1: Aggregate var_daily to desk level.
--         var_daily grain is desk + asset_class but pnl_attribution
--         is at desk level only. We take MAX VaR across asset classes
--         as a conservative desk-level estimate.
--         In production this would use a proper portfolio aggregation method.
desk_var AS (
  SELECT
    calculation_date,
    desk,
    MAX(var_97_5_usd) AS var_97_5_usd
  FROM market_risk_dev.gold.var_daily
  GROUP BY
    calculation_date,
    desk
),

-- Step 2: Get daily actual P&L from pnl_attribution.
--         Convert to actual_loss_usd: positive = loss, negative = gain.
--         This sign convention makes exception logic cleaner —
--         loss > VaR is a straightforward greater-than comparison.
daily_pnl AS (
  SELECT
    pnl_date,
    desk,
    -actual_pnl_usd AS actual_loss_usd
  FROM market_risk_dev.gold.pnl_attribution
),

-- Step 3: Join VaR to actual P&L and identify exceptions.
--         An exception = actual_loss_usd > var_97_5_usd.
--         exception_magnitude shows severity of the model miss.
exceptions_identified AS (
  SELECT
    pnl.pnl_date                                         AS backtest_date,
    pnl.desk,
    vr.var_97_5_usd,
    pnl.actual_loss_usd,
    CASE
      WHEN pnl.actual_loss_usd > vr.var_97_5_usd THEN TRUE
      ELSE FALSE
    END                                                  AS is_exception,
    CASE
      WHEN pnl.actual_loss_usd > vr.var_97_5_usd
      THEN ROUND(pnl.actual_loss_usd - vr.var_97_5_usd, 2)
      ELSE NULL
    END                                                  AS exception_magnitude
  FROM daily_pnl        pnl
  INNER JOIN desk_var   vr
    ON  pnl.desk     = vr.desk
    AND pnl.pnl_date = vr.calculation_date
),

-- Step 4: Calculate rolling 250-day exception count per desk.
--         SUM() OVER with ROWS BETWEEN counts exceptions in the 249 rows
--         before plus the current row = 250-day rolling window.
--         This is exactly the window the Basel Committee specifies
--         for the backtesting traffic light assessment.
rolling_exceptions AS (
  SELECT
    backtest_date,
    desk,
    var_97_5_usd,
    actual_loss_usd,
    is_exception,
    exception_magnitude,
    SUM(CAST(is_exception AS INT)) OVER (
      PARTITION BY desk
      ORDER BY backtest_date ASC
      ROWS BETWEEN 249 PRECEDING AND CURRENT ROW
    )                                                    AS rolling_250d_exceptions
  FROM exceptions_identified
),

-- Step 5: Apply Basel traffic light zones.
--         Thresholds defined once here — single place to update
--         if regulatory requirements change.
basel_assessment AS (
  SELECT
    backtest_date,
    desk,
    var_97_5_usd,
    actual_loss_usd,
    is_exception,
    exception_magnitude,
    rolling_250d_exceptions,
    CASE
      WHEN rolling_250d_exceptions >= 10 THEN 'RED'
      WHEN rolling_250d_exceptions >=  5 THEN 'AMBER'
      ELSE                                    'GREEN'
    END                                                  AS basel_zone,
    CASE
      WHEN rolling_250d_exceptions >= 10 THEN 'REJECTED'
      WHEN rolling_250d_exceptions >=  5 THEN 'REVIEW'
      ELSE                                    'ACCEPTED'
    END                                                  AS model_status
  FROM rolling_exceptions
)

SELECT
  backtest_date,
  desk,
  ROUND(var_97_5_usd, 2)              AS var_97_5_usd,
  ROUND(actual_loss_usd, 2)           AS actual_loss_usd,
  is_exception,
  exception_magnitude,
  CAST(rolling_250d_exceptions AS INT) AS rolling_250d_exceptions,
  basel_zone,
  model_status,
  current_timestamp()                 AS _gold_loaded_at
FROM basel_assessment
) AS source
ON target.backtest_date = source.backtest_date
AND target.desk = source.desk
WHEN MATCHED THEN 
	UPDATE SET 
		target.var_97_5_usd 			= source.var_97_5_usd,
		target.actual_loss_usd 			= source.actual_loss_usd,
		target.is_exception 			= source.is_exception,
		target.exception_magnitude 		= source.exception_magnitude,
		target.rolling_250d_exceptions  = source.rolling_250d_exceptions,
		target.basel_zone 				= source.basel_zone,
		target.model_status 			= source.model_status,
		target._gold_loaded_at 			= source._gold_loaded_at
WHEN NOT MATCHED THEN 
	INSERT (
    backtest_date,
    desk,
		var_97_5_usd,
		actual_loss_usd,
		is_exception,
		exception_magnitude,
		rolling_250d_exceptions,
		basel_zone,
		model_status,
		_gold_loaded_at
	)
	VALUES (
    source.backtest_date,
    source.desk,
		source.var_97_5_usd,	
		source.actual_loss_usd,
		source.is_exception,
		source.exception_magnitude,
		source.rolling_250d_exceptions,
		source.basel_zone,
		source.model_status,
		source._gold_loaded_at
	)
;

In [0]:
-- Data Quality check for gold.var_backtest
WITH quality_summary AS (
  SELECT
    COUNT(*)                                                      AS total_rows,
    COUNT(DISTINCT desk)                                          AS unique_desks,
    COUNT(DISTINCT backtest_date)                                 AS unique_dates,
    COUNT(CASE WHEN var_97_5_usd            IS NULL THEN 1 END)   AS null_var,
    COUNT(CASE WHEN actual_loss_usd         IS NULL THEN 1 END)   AS null_actual_loss,
    COUNT(CASE WHEN rolling_250d_exceptions IS NULL THEN 1 END)   AS null_rolling_count,
    COUNT(CASE WHEN basel_zone              IS NULL THEN 1 END)   AS null_basel_zone,
    -- exception_magnitude must be null when is_exception = false
    COUNT(
      CASE WHEN is_exception = FALSE
            AND exception_magnitude IS NOT NULL
           THEN 1 END
    )                                                             AS magnitude_on_non_exception,
    -- exception_magnitude must be positive when is_exception = true
    COUNT(
      CASE WHEN is_exception = TRUE
            AND (exception_magnitude IS NULL OR exception_magnitude <= 0)
           THEN 1 END
    )                                                             AS missing_magnitude_on_exception,
    -- Basel zone consistency checks
    COUNT(
      CASE WHEN basel_zone = 'RED'
            AND rolling_250d_exceptions < 10 THEN 1 END
    )                                                             AS red_zone_mismatch,
    COUNT(
      CASE WHEN basel_zone = 'AMBER'
            AND (rolling_250d_exceptions < 5
              OR rolling_250d_exceptions >= 10) THEN 1 END
    )                                                             AS amber_zone_mismatch,
    -- Exception statistics
    COUNT(CASE WHEN is_exception = TRUE  THEN 1 END)              AS total_exceptions,
    COUNT(CASE WHEN is_exception = FALSE THEN 1 END)              AS total_non_exceptions,
    COUNT(CASE WHEN basel_zone = 'GREEN' THEN 1 END)              AS green_rows,
    COUNT(CASE WHEN basel_zone = 'AMBER' THEN 1 END)              AS amber_rows,
    COUNT(CASE WHEN basel_zone = 'RED'   THEN 1 END)              AS red_rows
  FROM market_risk_dev.gold.var_backtest
)

SELECT
  total_rows,
  unique_desks,
  unique_dates,
  null_var,
  null_actual_loss,
  null_rolling_count,
  null_basel_zone,
  magnitude_on_non_exception,
  missing_magnitude_on_exception,
  red_zone_mismatch,
  amber_zone_mismatch,
  total_exceptions,
  total_non_exceptions,
  green_rows,
  amber_rows,
  red_rows,
  CASE
    WHEN null_var                       > 0 THEN 'FAIL — null var found'
    WHEN null_actual_loss               > 0 THEN 'FAIL — null actual_loss found'
    WHEN null_rolling_count             > 0 THEN 'FAIL — null rolling count found'
    WHEN null_basel_zone                > 0 THEN 'FAIL — null basel_zone found'
    WHEN magnitude_on_non_exception     > 0 THEN 'FAIL — magnitude present on non-exception row'
    WHEN missing_magnitude_on_exception > 0 THEN 'FAIL — magnitude missing on exception row'
    WHEN red_zone_mismatch              > 0 THEN 'FAIL — RED zone inconsistent with exception count'
    WHEN amber_zone_mismatch            > 0 THEN 'FAIL — AMBER zone inconsistent with exception count'
    ELSE 'PASS'
  END AS quality_status
FROM quality_summary;

In [0]:
-- ─────────────────────────────────────────────────────────────────────────────
-- gold.exposure_monitor
-- Purpose : Limit utilisation and breach monitoring per desk
-- Grain   : One row per desk per as_of_date
-- Source  : silver.positions_enriched + bronze.desk_limits
-- Status logic:
--   GREEN  — utilisation_pct < 80    — within safe operating range
--   AMBER  — utilisation_pct >= 80
--             and < 100              — approaching limit, early warning
--   RED    — utilisation_pct >= 100  — limit breached, immediate action required
-- ─────────────────────────────────────────────────────────────────────────────
CREATE OR REPLACE TABLE market_risk_dev.gold.exposure_monitor (
  as_of_date            DATE    NOT NULL COMMENT 'Date exposure was calculated',
  desk                  STRING  NOT NULL COMMENT 'Trading desk name',
  gross_exposure_usd    DOUBLE  NOT NULL COMMENT 'Sum of absolute notional values across all positions regardless of direction',
  net_exposure_usd      DOUBLE  NOT NULL COMMENT 'Sum of signed exposures — LONG positive, SHORT negative',
  limit_usd             DOUBLE  NOT NULL COMMENT 'Risk limit for this desk in USD from bronze.desk_limits',
  utilisation_pct       DOUBLE  NOT NULL COMMENT 'Gross exposure as percentage of limit. 100 = at limit. Above 100 = breach.',
  limit_status          STRING  NOT NULL COMMENT 'Traffic light: GREEN = below 80%, AMBER = 80<=100%, RED = >= 100%',
  warning_flag          BOOLEAN NOT NULL COMMENT 'True when utilisation_pct >= 80 and < 100 — approaching limit',
  breach_flag           BOOLEAN NOT NULL COMMENT 'True when utilisation_pct >= 100 — limit breached',
  long_exposure_usd     DOUBLE  NOT NULL COMMENT 'Total exposure from LONG positions only',
  short_exposure_usd    DOUBLE  NOT NULL COMMENT 'Total exposure from SHORT positions only',
  position_count        INT     NOT NULL COMMENT 'Total number of positions for this desk',
  _gold_loaded_at       TIMESTAMP        COMMENT 'Timestamp when row was loaded into Gold'
)
USING DELTA
COMMENT 'Daily limit utilisation and breach monitoring by desk. Traffic light status (GREEN/AMBER/RED) for reporting layer. One row per desk per as_of_date.'
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite'   = 'true',
  'delta.autoOptimize.autoCompact'     = 'true',
  'delta.logRetentionDuration'         = 'interval 90 days',
  'delta.deletedFileRetentionDuration' = 'interval 90 days'
);

In [0]:
MERGE INTO market_risk_dev.gold.exposure_monitor AS target
USING (
WITH

-- Step 1: Aggregate positions to desk level.
--         Gross exposure = sum of absolute notional values.
--         Most risk limits are measured against gross exposure because
--         SHORT positions consume limit capacity the same as LONG positions.
desk_exposure AS (
  SELECT
    desk,
    COUNT(DISTINCT trade_id)                              AS position_count,
    ROUND(SUM(ABS(notional_usd)), 2)                     AS gross_exposure_usd,
    ROUND(SUM(net_exposure_usd), 2)                      AS net_exposure_usd,
    ROUND(
      SUM(CASE WHEN direction = 'LONG'
               THEN notional_usd ELSE 0 END),
      2
    )                                                     AS long_exposure_usd,
    ROUND(
      SUM(CASE WHEN direction = 'SHORT'
               THEN notional_usd ELSE 0 END),
      2
    )                                                     AS short_exposure_usd
  FROM market_risk_dev.silver.positions_enriched
  GROUP BY desk
),

-- Step 2: Join desk limits from Bronze reference table.
--         INNER JOIN because every desk must have a limit defined.
--         A missing limit is a data quality problem that should surface
--         as a missing row rather than a null utilisation figure.
exposure_with_limits AS (
  SELECT
    de.desk,
    de.position_count,
    de.gross_exposure_usd,
    de.net_exposure_usd,
    de.long_exposure_usd,
    de.short_exposure_usd,
    CAST(dl.limit_usd AS DOUBLE)                          AS limit_usd,
    ROUND(
      (de.gross_exposure_usd / CAST(dl.limit_usd AS DOUBLE)) * 100,
      2
    )                                                     AS utilisation_pct
  FROM desk_exposure                            de
  INNER JOIN market_risk_dev.bronze.desk_limits dl
    ON UPPER(de.desk) = UPPER(dl.desk)
),

-- Step 3: Apply traffic light logic.
--         All three status columns derived from utilisation_pct in one CTE
--         so thresholds are defined once — single place to update if
--         business rules change (e.g. warning threshold moves from 80% to 75%).
status_applied AS (
  SELECT
    desk,
    position_count,
    gross_exposure_usd,
    net_exposure_usd,
    limit_usd,
    utilisation_pct,
    long_exposure_usd,
    short_exposure_usd,
    CASE
      WHEN utilisation_pct >= 100 THEN 'RED'
      WHEN utilisation_pct >=  80 THEN 'AMBER'
      ELSE                             'GREEN'
    END                                                   AS limit_status,
    CASE
      WHEN utilisation_pct >= 80
       AND utilisation_pct <  100 THEN TRUE
      ELSE FALSE
    END                                                   AS warning_flag,
    CASE
      WHEN utilisation_pct >= 100 THEN TRUE
      ELSE FALSE
    END                                                   AS breach_flag
  FROM exposure_with_limits
)

SELECT
  CURRENT_DATE()                  AS as_of_date,
  desk,
  gross_exposure_usd,
  net_exposure_usd,
  limit_usd,
  utilisation_pct,
  limit_status,
  warning_flag,
  breach_flag,
  long_exposure_usd,
  short_exposure_usd,
  CAST(position_count AS INT)     AS position_count,
  current_timestamp()             AS _gold_loaded_at
FROM status_applied
) AS source
ON  target.desk = source.desk
AND target.as_of_date = source.as_of_date
WHEN MATCHED THEN 
	UPDATE SET 
		target.gross_exposure_usd = source.gross_exposure_usd,
		target.net_exposure_usd   = source.net_exposure_usd,  
		target.limit_usd          = source.limit_usd,         
		target.utilisation_pct    = source.utilisation_pct,   
		target.limit_status       = source.limit_status,      
		target.warning_flag       = source.warning_flag,      
		target.breach_flag        = source.breach_flag,       
		target.long_exposure_usd  = source.long_exposure_usd, 
		target.short_exposure_usd = source.short_exposure_usd,
		target.position_count     = source.position_count,    
		target._gold_loaded_at    = source._gold_loaded_at
WHEN NOT MATCHED THEN 
	INSERT (
		as_of_date,        	
		desk,              
		gross_exposure_usd,
		net_exposure_usd,  
		limit_usd,         
		utilisation_pct,   
		limit_status,      
		warning_flag,      
		breach_flag,       
		long_exposure_usd, 
		short_exposure_usd,
		position_count,    
		_gold_loaded_at
	)
	VALUES (
		source.as_of_date,        	
		source.desk,              		
		source.gross_exposure_usd,
		source.net_exposure_usd,  
		source.limit_usd,         
		source.utilisation_pct,   
		source.limit_status,      
		source.warning_flag,      
		source.breach_flag,       
		source.long_exposure_usd, 
		source.short_exposure_usd,
		source.position_count,    
		source._gold_loaded_at
	)
;

In [0]:
-- Data quality for gold.exposure_monitor table
WITH quality_summary AS (
  SELECT
    COUNT(*)                                                      AS total_rows,
    COUNT(DISTINCT desk)                                          AS unique_desks,
    COUNT(CASE WHEN gross_exposure_usd IS NULL THEN 1 END)        AS null_gross_exposure,
    COUNT(CASE WHEN limit_usd          IS NULL THEN 1 END)        AS null_limit,
    COUNT(CASE WHEN utilisation_pct    IS NULL THEN 1 END)        AS null_utilisation,
    COUNT(CASE WHEN limit_status       IS NULL THEN 1 END)        AS null_limit_status,
    COUNT(CASE WHEN gross_exposure_usd  < 0    THEN 1 END)        AS negative_gross_exposure,
    COUNT(CASE WHEN limit_usd          <= 0    THEN 1 END)        AS non_positive_limit,
    COUNT(CASE WHEN utilisation_pct     < 0    THEN 1 END)        AS negative_utilisation,
    -- RED must have breach_flag = true and warning_flag = false
    COUNT(
      CASE WHEN limit_status = 'RED'
            AND (breach_flag = FALSE OR warning_flag = TRUE)
           THEN 1 END
    )                                                             AS red_flag_mismatch,
    -- AMBER must have warning_flag = true and breach_flag = false
    COUNT(
      CASE WHEN limit_status = 'AMBER'
            AND (warning_flag = FALSE OR breach_flag = TRUE)
           THEN 1 END
    )                                                             AS amber_flag_mismatch,
    -- GREEN must have both flags false
    COUNT(
      CASE WHEN limit_status = 'GREEN'
            AND (warning_flag = TRUE OR breach_flag = TRUE)
           THEN 1 END
    )                                                             AS green_flag_mismatch,
    -- limit_status must only contain valid values
    COUNT(
      CASE WHEN limit_status NOT IN ('RED', 'AMBER', 'GREEN')
           THEN 1 END
    )                                                             AS invalid_status_values,
    COUNT(CASE WHEN limit_status = 'RED'   THEN 1 END)            AS red_desks,
    COUNT(CASE WHEN limit_status = 'AMBER' THEN 1 END)            AS amber_desks,
    COUNT(CASE WHEN limit_status = 'GREEN' THEN 1 END)            AS green_desks
  FROM market_risk_dev.gold.exposure_monitor
)

SELECT
  total_rows,
  unique_desks,
  null_gross_exposure,
  null_limit,
  null_utilisation,
  null_limit_status,
  negative_gross_exposure,
  non_positive_limit,
  negative_utilisation,
  red_flag_mismatch,
  amber_flag_mismatch,
  green_flag_mismatch,
  invalid_status_values,
  red_desks,
  amber_desks,
  green_desks,
  CASE
    WHEN null_gross_exposure     > 0 THEN 'FAIL — null gross_exposure found'
    WHEN null_limit              > 0 THEN 'FAIL — null limit found'
    WHEN null_utilisation        > 0 THEN 'FAIL — null utilisation found'
    WHEN null_limit_status       > 0 THEN 'FAIL — null limit_status found'
    WHEN negative_gross_exposure > 0 THEN 'FAIL — negative gross exposure found'
    WHEN non_positive_limit      > 0 THEN 'FAIL — non-positive limit found'
    WHEN red_flag_mismatch       > 0 THEN 'FAIL — RED status inconsistent with flags'
    WHEN amber_flag_mismatch     > 0 THEN 'FAIL — AMBER status inconsistent with flags'
    WHEN green_flag_mismatch     > 0 THEN 'FAIL — GREEN status inconsistent with flags'
    WHEN invalid_status_values   > 0 THEN 'FAIL — invalid limit_status value found'
    ELSE 'PASS'
  END AS quality_status
FROM quality_summary;

In [0]:
-- High level summary for the gold/curated tables 
SELECT
  'var_daily'        AS table_name,
  COUNT(*)           AS row_count,
  COUNT(DISTINCT desk) AS unique_desks,
  MAX(calculation_date) AS latest_date
FROM market_risk_dev.gold.var_daily

UNION ALL

SELECT
  'var_backtest'     AS table_name,
  COUNT(*)           AS row_count,
  COUNT(DISTINCT desk) AS unique_desks,
  MAX(backtest_date) AS latest_date
FROM market_risk_dev.gold.var_backtest

UNION ALL

SELECT
  'pnl_attribution'  AS table_name,
  COUNT(*)           AS row_count,
  COUNT(DISTINCT desk) AS unique_desks,
  MAX(pnl_date)      AS latest_date
FROM market_risk_dev.gold.pnl_attribution

UNION ALL

SELECT
  'exposure_monitor' AS table_name,
  COUNT(*)           AS row_count,
  COUNT(DISTINCT desk) AS unique_desks,
  MAX(as_of_date)    AS latest_date
FROM market_risk_dev.gold.exposure_monitor

ORDER BY table_name;